In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

acc = pd.read_csv("data_cog/acc_data.csv")

print(acc.shape)
print(acc['timestamp'].min())
print(acc['timestamp'].max())
print(acc['id'].nunique()) 


print(len(acc))

In [ ]:
acc['timestamp_date'] = pd.to_datetime(acc['timestamp'])
acc['date'] = acc['timestamp_date'].dt.date

acc = acc.sort_values(by=['id', 'timestamp_date']).reset_index(drop=True)

# caluclates the difference between each row and converts those time gaps into seconds then minutes
acc['time_diff'] = acc.groupby('id')['timestamp_date'].diff().dt.total_seconds().div(60)

gaps = acc[acc['time_diff']>1.5] #acc[] twice cause it only keeps the ones with true and drops the false
print(len(gaps))

In [ ]:
acc['date'].tail(70)
acc['id'].agg(['min', 'max'])

In [ ]:
#feature extraction

daily_intensity = (acc.groupby(['id', 'date']).agg(
    total_activityscore = ('activityscore', 'sum'),
    mean_activityscore = ('activityscore', 'mean'),
    valid_minutes = ('timestamp_date', 'count'),
    std_activityscore = ('activityscore', 'std')
    ).reset_index()
)   

#flag days with at least 80% of the day
daily_intensity['complete_day'] = daily_intensity['valid_minutes'] >= 1440 * 0.8

#-----------------------

daily_transitions = (
    acc.groupby(["id", "date"])
       .agg(
           sedentary_to_upright_transitions=("sed2upr", "sum"),
           upright_to_sedentary_transitions=("upr2sed", "sum")
       )
       .reset_index()
)

acc["is_sedentary"] = (
    acc["pa"].astype(str).str.lower().eq("sed")
)

# Check whether timestamps are consecutive
acc["consecutive_minute"] = acc["time_diff"].eq(1)

# ----------------- MAX SEDENTARY BOUT -----------------

acc["awake_sedentary"] = (
    acc["is_sedentary"]
    & acc["sleep"].eq(0)
)

acc["previous_awake_sedentary"] = (
    acc.groupby(["id", "date"])["awake_sedentary"].shift(1)
)

acc["new_bout"] = (
    ~acc["awake_sedentary"]
    | ~acc["previous_awake_sedentary"].fillna(False)
    | ~acc["consecutive_minute"]
)

acc["bout_id"] = (
    acc.groupby(["id", "date"])["new_bout"].cumsum()
)

# Only keep rows where awake_sedentary is True
sedentary_rows = acc.loc[acc["awake_sedentary"]].copy()

# Length of every sedentary bout in minutes
sedentary_bouts = (
    sedentary_rows
    .groupby(["id", "date", "bout_id"])
    .size()
    .reset_index(name="bout_minutes")
)

# Longest sedentary bout for each person on each day
max_sedentary_bout = (
    sedentary_bouts
    .groupby(["id", "date"])["bout_minutes"]
    .max()
    .reset_index(name="max_sedentary_bout_minutes")
)

#----------------- DAILY BEHAVIOR -----------------
daily_behavior = (
    acc.groupby(["id", "date"])
       .agg(
           total_steps=("stepcount", "sum"),
           mvpa_steps=("stepcount_mvpa", "sum"),
           sleep_minutes=("sleep", "sum"),
           upright_seconds=("uprtime", "sum"),
           stepping_seconds=("steptime", "sum"),
           sedentary_minutes=("awake_sedentary", "sum"),
           valid_sleep_observations=("sleep", "count")
       )
       .reset_index()
)

daily_behavior["upright_minutes"] = (
    daily_behavior["upright_seconds"] / 60
)

daily_behavior["stepping_minutes"] = (
    daily_behavior["stepping_seconds"] / 60
)

daily_behavior = daily_behavior.drop(
    columns=["upright_seconds", "stepping_seconds"]
)



# ----------------- MERGE EVERYTHING -----------------

daily_acc = (
    daily_intensity
    .merge(
        daily_transitions,
        on=["id", "date"],
        how="left"
    )
    .merge(
        max_sedentary_bout,
        on=["id", "date"],
        how="left"
    )
    .merge(
        daily_behavior,
        on=["id", "date"],
        how="left"
    )
)

#fixing missing values in sleep
daily_acc["sleep_coverage"] = (daily_acc["valid_sleep_observations"] / daily_acc["valid_minutes"])

# Days with no sedentary bout should receive 0
daily_acc["max_sedentary_bout_minutes"] = (
    daily_acc["max_sedentary_bout_minutes"].fillna(0)
)

#if my sleep data is incomplete then the below columns should be set to NaN - otherwise it will be misleading to use those values in analysis
daily_acc.loc[
    daily_acc["sleep_coverage"] < 0.80,
    [   
        "sleep_minutes",
        "sedentary_minutes",
        "max_sedentary_bout_minutes"
    ]
] = np.nan




#only keeping the complete days
daily_acc = daily_acc.loc[
    daily_acc["complete_day"]
].copy()


In [ ]:
print(daily_acc.shape)
print(daily_acc.head())
print(daily_acc.isna().sum()) 
print(daily_acc.describe())

print(acc[["uprtime", "steptime", "sleep"]].head(20))
print(acc[["uprtime", "steptime", "sleep"]].describe())


In [ ]:
print(len(acc))
#sleep has missing values 
#there are days with less than 34 minutes and those with more than 1500 


In [ ]:
day_counts = (
    acc.groupby(["id", "date"])
    .size()
    .reset_index(name="minutes")
)

#confirmed they are just day-time savings stuff
over_1440 = day_counts.loc[
    day_counts["minutes"] > 1440
]

print(over_1440)

In [ ]:
#Step 4

days_per_id = daily_acc.groupby("id").size().sort_values()

days_per_id.head(56)

In [ ]:
#temporary invalid day rule
daily_acc["probable_nonwear"] = (
    (daily_acc["total_steps"] == 0)
    & (daily_acc["stepping_minutes"] == 0)
    &(daily_acc["sleep_minutes"] == 0)
    & (daily_acc["sedentary_to_upright_transitions"] == 0)
    & (daily_acc["std_activityscore"] == 0)
    & (daily_acc["valid_minutes"] >= 1440*0.8)
    & (daily_acc["mean_activityscore"] <= 80) 
)

print(
    daily_acc.loc[
        daily_acc["probable_nonwear"],
        "mean_activityscore"
    ].value_counts()
)

print(
    daily_acc.loc[daily_acc["probable_nonwear"]]
    .groupby("id")
    .size()
    .sort_values(ascending=False)
)

In [ ]:
missing_stepcount = acc.groupby(['id','date'])['stepcount'].apply(lambda x: x.isna().sum()).reset_index(name='missing_stepcount')

missing_stepcount_204 = missing_stepcount.loc[missing_stepcount["id"] == 204].sort_values(by="missing_stepcount",ascending=False)

print(missing_stepcount_204)

zero_step_days = daily_acc.loc[daily_acc["total_steps"] == 0]

number_of = zero_step_days.groupby("id").size().sort_values(ascending=False)

print(number_of)

sedentary_steps_1440 = daily_acc.loc[daily_acc["sedentary_minutes"] == 1440]

print(sedentary_steps_1440.groupby("id").size().sort_values(ascending=False))

#this means: all rows had pa = "sed" and sleep = "0"
#7/13 sed for 204

participant_check = (
    daily_acc.loc[daily_acc["id"].isin([204, 206])]
    .groupby("id")
    .agg(
        total_days=("date", "size"),
        invalid_days=("probable_nonwear", "sum")
    )
)

participant_check["valid_days"] = (
    participant_check["total_days"]
    - participant_check["invalid_days"]
)

print(participant_check)

print(
    daily_acc.loc[
        daily_acc["id"] == 206
    ].to_string(index=False)
)

In [ ]:
# Preserve the original daily dataset
daily_acc_original = daily_acc.copy()

daily_acc = daily_acc.loc[
    daily_acc["valid_minutes"].between(1152, 1440)
].copy()

# Remove probable_nonwear days where probable_nonwear is True
daily_acc = daily_acc.loc[
    ~daily_acc["probable_nonwear"]
].copy()

# Remove participant 204 entirely
daily_acc = daily_acc.loc[
    daily_acc["id"] != 204
].copy()

print("Original rows:", len(daily_acc_original))
print("Cleaned rows:", len(daily_acc))

#Participant 204 was excluded because most or all of their recorded days showed a constant probable-nonwear pattern. For other participants, only affected participant-days were excluded.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

features = [
    "mean_activityscore",
    "std_activityscore",
    "total_steps",
    "mvpa_steps",
    "sedentary_to_upright_transitions",
    "max_sedentary_bout_minutes",
    "sedentary_minutes",
    "upright_minutes",
    "stepping_minutes"
]

for column in features:
    sns.histplot(daily_acc[column].dropna(), kde=True)
    plt.title(column)
    plt.show()

In [ ]:
#Corr matrix
correlation_features = [
    "mean_activityscore",
    "std_activityscore",
    "total_steps",
    "mvpa_steps",
    "sedentary_to_upright_transitions",
    "upright_to_sedentary_transitions",
    "max_sedentary_bout_minutes",
    "sedentary_minutes",
    "upright_minutes",
    "stepping_minutes",
    "sleep_minutes"
]

correlation_matrix = daily_acc[correlation_features].corr(
    method="spearman"
)

plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
final_features = [
    "total_steps",                       # overall movement
    "mvpa_steps",                        # movement intensity
    "sedentary_to_upright_transitions",  # activity fragmentation
    "sedentary_minutes",                 # total sedentary behavior
    "max_sedentary_bout_minutes",        # prolonged sedentary behavior
    "sleep_minutes"                      # sleep
]

In [ ]:
# Make sure date is datetime
daily_acc["date"] = pd.to_datetime(
    daily_acc["date"],
    errors="coerce"
).dt.normalize()

# Sort before creating temporal variables
acc_temporal = (
    daily_acc
    .sort_values(["id", "date"])
    .reset_index(drop=True)
    .copy()
)

# Behavior from previous available accelerometer observation

acc_temporal["prior_available_mvpa_steps"] = (
    acc_temporal.groupby("id")["mvpa_steps"].shift(1)
)

acc_temporal["prior_available_transitions"] = (
    acc_temporal.groupby("id")[
        "sedentary_to_upright_transitions"
    ].shift(1)
)

acc_temporal["prior_available_max_sedentary_bout"] = (
    acc_temporal.groupby("id")[
        "max_sedentary_bout_minutes"
    ].shift(1)
)
acc_temporal["prior_available_steps"] = (
    acc_temporal
    .groupby("id")["total_steps"]
    .shift(1)
)

acc_temporal["prior_available_sedentary_minutes"] = (
    acc_temporal
    .groupby("id")["sedentary_minutes"]
    .shift(1)
)

acc_temporal["prior_available_sleep_minutes"] = (
    acc_temporal
    .groupby("id")["sleep_minutes"]
    .shift(1)
)

# Calendar gap from previous available accelerometer observation
acc_temporal["days_since_prior_acc"] = (
    acc_temporal
    .groupby("id")["date"]
    .diff()
    .dt.days
)

In [ ]:
# ----------------- MERGING -----------------

healthy = pd.read_csv("data/healthy_processed.csv") #healthy data is already processed, cleaned, and includes temporal variables, so we can use it directly

healthy["date"] = pd.to_datetime(
    healthy["date"],
    errors="coerce"
).dt.normalize()

acc_temporal["date"] = pd.to_datetime(
    acc_temporal["date"],
    errors="coerce"
).dt.normalize()

print(
    "Healthy duplicates:",
    healthy.duplicated(["id", "date"]).sum()
)

print(
    "Accelerometer duplicates:",
    acc_temporal.duplicated(["id", "date"]).sum()
)

merged_data = healthy.merge(
    acc_temporal,
    on=["id", "date"],
    how="inner"
)

merged_data = (
    merged_data
    .sort_values(["id", "date"])
    .reset_index(drop=True)
)

print("Healthy rows:", len(healthy))
print("Accelerometer rows:", len(acc_temporal))
print("Merged rows:", len(merged_data))
print("Merged participants:", merged_data["id"].nunique())

In [ ]:
print([
    column for column in merged_data.columns
    if "prior" in column
])

In [ ]:
temporal_columns = [
    "prior_3day_bad_mood",
    "prior_cognition",
    "prior_available_mvpa_steps",
    "prior_available_transitions",
    "prior_available_max_sedentary_bout",
    "prior_available_steps",
    "prior_available_sedentary_minutes",
    "prior_available_sleep_minutes"
]

print(
    merged_data[temporal_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
main_model_data = merged_data.dropna(
    subset=[
        "prior_3day_bad_mood",
        "mindsharp_z",
        "prior_cognition",
        "prior_available_steps"

    ]
)

print(len(main_model_data))
print(main_model_data["id"].nunique())

In [ ]:
import statsmodels.formula.api as smf

steps_data = merged_data.dropna(
    subset=[
        "total_steps",
        "prior_3day_bad_mood",
        "prior_available_steps",
        "id"
    ]
).copy()

steps_model = smf.mixedlm(
    "total_steps ~ prior_3day_bad_mood + prior_available_steps",
    data=steps_data,
    groups=steps_data["id"]
)

steps_result = steps_model.fit(
    reml=False,
    method="lbfgs"
)

print(steps_result.summary())
print("Converged:", steps_result.converged)

In [ ]:
sedentary_data = merged_data.dropna(
    subset=[
        "sedentary_minutes",
        "prior_3day_bad_mood",
        "prior_available_sedentary_minutes",
        "id"
    ]
).copy()

sedentary_model = smf.mixedlm(
    """
    sedentary_minutes
    ~ prior_3day_bad_mood
    + prior_available_sedentary_minutes
    """,
    data=sedentary_data,
    groups=sedentary_data["id"]
)

sedentary_result = sedentary_model.fit(
    reml=False,
    method="lbfgs"
)

print(sedentary_result.summary())
print("Converged:", sedentary_result.converged)

In [ ]:
bout_data = merged_data.dropna(
    subset=[
        "max_sedentary_bout_minutes",
        "prior_3day_bad_mood",
        "prior_available_max_sedentary_bout",
        "id"
    ]
).copy()

bout_model = smf.mixedlm(
    """
    max_sedentary_bout_minutes
    ~ prior_3day_bad_mood
    + prior_available_max_sedentary_bout
    """,
    data=bout_data,
    groups=bout_data["id"]
)

bout_result = bout_model.fit(
    reml=False,
    method="lbfgs"
)

print(bout_result.summary())
print("Converged:", bout_result.converged)

In [ ]:
mvpa_data = merged_data.dropna(
    subset=[
        "mvpa_steps",
        "prior_3day_bad_mood",
        "prior_available_mvpa_steps",
        "id"
    ]
).copy()

mvpa_model = smf.mixedlm(
    """
    mvpa_steps
    ~ prior_3day_bad_mood
    + prior_available_mvpa_steps
    """,
    data=mvpa_data,
    groups=mvpa_data["id"]
)

mvpa_result = mvpa_model.fit(
    reml=False,
    method="lbfgs"
)

print(mvpa_result.summary())
print("Converged:", mvpa_result.converged)

In [ ]:
transition_data = merged_data.dropna(
    subset=[
        "sedentary_to_upright_transitions",
        "prior_3day_bad_mood",
        "prior_available_transitions",
        "id"
    ]
).copy()

transition_model = smf.mixedlm(
    """
    sedentary_to_upright_transitions
    ~ prior_3day_bad_mood
    + prior_available_transitions
    """,
    data=transition_data,
    groups=transition_data["id"]
)

transition_result = transition_model.fit(
    reml=False,
    method="lbfgs"
)

print(transition_result.summary())
print("Converged:", transition_result.converged)

In [ ]:
sleep_data = merged_data.dropna(
    subset=[
        "sleep_minutes",
        "prior_3day_bad_mood",
        "prior_available_sleep_minutes",
        "id"
    ]
).copy()

sleep_model = smf.mixedlm(
    """
    sleep_minutes
    ~ prior_3day_bad_mood
    + prior_available_sleep_minutes
    """,
    data=sleep_data,
    groups=sleep_data["id"]
)

sleep_result = sleep_model.fit(
    reml=False,
    method="lbfgs"
)

print(sleep_result.summary())
print("Converged:", sleep_result.converged)

In [ ]:
#Step8   

In [ ]:
cognition_steps_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "prior_available_steps",
        "prior_cognition",
        "id"
    ]
).copy()

cognition_steps_model = smf.mixedlm(
    """
    mindsharp_z
    ~ prior_available_steps
    + prior_cognition
    """,
    data=cognition_steps_data,
    groups=cognition_steps_data["id"]
)

cognition_steps_result = cognition_steps_model.fit(
    reml=False,
    method="lbfgs"
)

print(cognition_steps_result.summary())
print("Converged:", cognition_steps_result.converged)

In [ ]:
cognition_sedentary_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "prior_available_sedentary_minutes",
        "prior_cognition",
        "id"
    ]
).copy()

cognition_sedentary_model = smf.mixedlm(
    """
    mindsharp_z
    ~ prior_available_sedentary_minutes
    + prior_cognition
    """,
    data=cognition_sedentary_data,
    groups=cognition_sedentary_data["id"]
)

cognition_sedentary_result = cognition_sedentary_model.fit(
    reml=False,
    method="lbfgs"
)

print(cognition_sedentary_result.summary())
print("Converged:", cognition_sedentary_result.converged)

In [ ]:
cognition_bout_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "prior_available_max_sedentary_bout",
        "prior_cognition",
        "id"
    ]
).copy()

cognition_bout_model = smf.mixedlm(
    """
    mindsharp_z
    ~ prior_available_max_sedentary_bout
    + prior_cognition
    """,
    data=cognition_bout_data,
    groups=cognition_bout_data["id"]
)

cognition_bout_result = cognition_bout_model.fit(
    reml=False,
    method="lbfgs"
)

print(cognition_bout_result.summary())
print("Converged:", cognition_bout_result.converged)

In [ ]:
cognition_mvpa_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "prior_available_mvpa_steps",
        "prior_cognition",
        "id"
    ]
).copy()

cognition_mvpa_model = smf.mixedlm(
    """
    mindsharp_z
    ~ prior_available_mvpa_steps
    + prior_cognition
    """,
    data=cognition_mvpa_data,
    groups=cognition_mvpa_data["id"]
)

cognition_mvpa_result = cognition_mvpa_model.fit(
    reml=False,
    method="lbfgs"
)

print(cognition_mvpa_result.summary())
print("Converged:", cognition_mvpa_result.converged)

In [ ]:
cognition_transition_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "prior_available_transitions",
        "prior_cognition",
        "id"
    ]
).copy()

cognition_transition_model = smf.mixedlm(
    """
    mindsharp_z
    ~ prior_available_transitions
    + prior_cognition
    """,
    data=cognition_transition_data,
    groups=cognition_transition_data["id"]
)

cognition_transition_result = cognition_transition_model.fit(
    reml=False,
    method="lbfgs"
)

print(cognition_transition_result.summary())
print("Converged:", cognition_transition_result.converged)

In [ ]:
cognition_sleep_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "prior_available_sleep_minutes",
        "prior_cognition",
        "id"
    ]
).copy()

cognition_sleep_model = smf.mixedlm(
    """
    mindsharp_z
    ~ prior_available_sleep_minutes
    + prior_cognition
    """,
    data=cognition_sleep_data,
    groups=cognition_sleep_data["id"]
)

cognition_sleep_result = cognition_sleep_model.fit(
    reml=False,
    method="lbfgs"
)

print(cognition_sleep_result.summary())
print("Converged:", cognition_sleep_result.converged)

#Clustered OLS gives another valid way to handle repeated observations when checking coefficient uncertainty.





In [ ]:
bout_ols = smf.ols(
    """
    mindsharp_z
    ~ prior_available_max_sedentary_bout
    + prior_cognition
    """,
    data=cognition_bout_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cognition_bout_data["id"]}
)

print(bout_ols.summary())

In [ ]:
sedentary_ols = smf.ols(
    """
    mindsharp_z
    ~ prior_available_sedentary_minutes
    + prior_cognition
    """,
    data=cognition_sedentary_data
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": cognition_sedentary_data["id"]
    }
)

print(sedentary_ols.summary())

In [ ]:
sleep_ols = smf.ols(
    """
    mindsharp_z
    ~ prior_available_sleep_minutes
    + prior_cognition
    """,
    data=cognition_sleep_data
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": cognition_sleep_data["id"]
    }
)

print(sleep_ols.summary())

In [ ]:
#This tests adjusted associations. It does not prove mediation because mood did not significantly predict behavior in Step 7.

behavior_variables = [
    "prior_available_steps",
    "prior_available_sedentary_minutes",
    "prior_available_max_sedentary_bout",
    "prior_available_mvpa_steps",
    "prior_available_transitions",
    "prior_available_sleep_minutes"
]

results = []

for behavior in behavior_variables:

    # Same observations must be used in both models
    model_data = merged_data.dropna(
        subset=[
            "mindsharp_z",
            "prior_3day_bad_mood",
            "prior_cognition",
            behavior,
            "id"
        ]
    ).copy()

    # Baseline model
    baseline_model = smf.ols(
        "mindsharp_z ~ prior_3day_bad_mood + prior_cognition",
        data=model_data
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": model_data["id"]}
    )

    # Combined mood + behavior model
    combined_model = smf.ols(
        f"""
        mindsharp_z
        ~ prior_3day_bad_mood
        + {behavior}
        + prior_cognition
        """,
        data=model_data
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": model_data["id"]}
    )

    results.append({
        "behavior": behavior,
        "n": len(model_data),
        "participants": model_data["id"].nunique(),

        "baseline_mood_beta":
            baseline_model.params["prior_3day_bad_mood"],

        "baseline_mood_p":
            baseline_model.pvalues["prior_3day_bad_mood"],

        "combined_mood_beta":
            combined_model.params["prior_3day_bad_mood"],

        "combined_mood_p":
            combined_model.pvalues["prior_3day_bad_mood"],

        "behavior_beta":
            combined_model.params[behavior],

        "behavior_p":
            combined_model.pvalues[behavior],

        "baseline_r_squared":
            baseline_model.rsquared,

        "combined_r_squared":
            combined_model.rsquared,

        "r_squared_change":
            combined_model.rsquared - baseline_model.rsquared
    })

pathway_results = pd.DataFrame(results)

print(pathway_results.round(4).to_string(index=False))

In [ ]:
merged_data["mood_person_mean"] = (
    merged_data.groupby("id")["prior_3day_bad_mood"]
    .transform("mean")
)

merged_data["mood_within"] = (
    merged_data["prior_3day_bad_mood"]
    - merged_data["mood_person_mean"]
)

merged_data["sleep_person_mean"] = (
    merged_data.groupby("id")["prior_available_sleep_minutes"]
    .transform("mean")
)

merged_data["sleep_within"] = (
    merged_data["prior_available_sleep_minutes"]
    - merged_data["sleep_person_mean"]
)


model_data = merged_data.dropna(
    subset=[
        "mindsharp_z",
        "mood_within",
        "mood_person_mean",
        "sleep_within",
        "sleep_person_mean",
        "prior_cognition",
        "id"
    ]
).copy()

model = smf.ols(
    """
    mindsharp_z
    ~ mood_within
    + mood_person_mean
    + sleep_within
    + sleep_person_mean
    + prior_cognition
    """,
    data=model_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_data["id"]}
)

print(model.summary())